In [22]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = '/content/drive/MyDrive/ML_final'
os.chdir(project_path)

import pandas as pd
import numpy as np
!pip install dagshub mlflow

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os
import dagshub
import mlflow

# 1. Set your credentials as environment variables
# Use your personal Access Token from DagsHub (Settings -> Apps -> Tokens)
os.environ['MLFLOW_TRACKING_USERNAME'] = 'ggzob23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '05414adcd0659c9178dccdd3fd05bd28837d5444'

# 2. Initialize with the project owner's repo details
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

# 3. Proceed as normal
mlflow.set_experiment('walmart-feature-engineering')

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI: https://dagshub.com/skupr23/ml_final.mlflow


In [24]:
import pandas as pd

# 1. Unzip the files into the 'data' folder
# The -o flag overwrites if you run this again, -q keeps it quiet
!unzip -o -q "data/train.csv.zip" -d "data/"
!unzip -o -q "data/test.csv.zip" -d "data/"
!unzip -o -q "data/features.csv.zip" -d "data/"

# 2. Check that the unzipped CSVs exist
print("Files in data folder:", os.listdir('data'))

# 3. Test loading the training data
try:
    train_df = pd.read_csv('data/train.csv')
    print("\nSuccess! Training data loaded.")
    print("Shape of training data:", train_df.shape)
    print(train_df.head())
except Exception as e:
    print(f"\nError loading file: {e}")

Files in data folder: ['test.csv.zip', 'train.csv.zip', 'sampleSubmission.csv.zip', 'features.csv.zip', 'stores.csv', 'train.csv', 'features.csv', 'test.csv', 'train_processed.pkl', 'test_processed.pkl']

Success! Training data loaded.
Shape of training data: (421570, 5)
   Store  Dept        Date  Weekly_Sales  IsHoliday
0      1     1  2010-02-05      24924.50      False
1      1     1  2010-02-12      46039.49       True
2      1     1  2010-02-19      41595.55      False
3      1     1  2010-02-26      19403.54      False
4      1     1  2010-03-05      21827.90      False


# Load raw data

In [25]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
features = pd.read_csv('data/features.csv')
stores = pd.read_csv('data/stores.csv')

for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(train.duplicated(['Store','Dept','Date']).sum())
print(test.duplicated(['Store','Dept','Date']).sum())

0
0


 # Merge safely (validate prevents silent row duplication)

In [26]:

train = (train.merge(features, on=['Store','Date','IsHoliday'], how='left', validate='many_to_one')
              .merge(stores, on='Store', how='left', validate='many_to_one'))
test = (test.merge(features, on=['Store','Date','IsHoliday'], how='left', validate='many_to_one')
             .merge(stores, on='Store', how='left', validate='many_to_one'))

train.shape, test.shape

((421570, 16), (115064, 15))

# Markdown missing indicators (missingness itself is informative)

In [27]:
markdown_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']

for df in [train, test]:
    for col in markdown_cols:
        df[f'{col}_missing'] = df[col].isna().astype('int8')
        df[col] = df[col].fillna(0)
    df['markdown_total'] = df[markdown_cols].sum(axis=1)
    df['markdown_count'] = (df[markdown_cols] > 0).sum(axis=1)
    df['has_any_markdown'] = (df['markdown_total'] > 0).astype('int8')

# CPI / Unemployment: forward-fill using train+test combined so test inherits last known train value

In [28]:
train['source'] = 'train'
test['source'] = 'test'
combined = pd.concat([train, test], ignore_index=True).sort_values(['Store','Date'])

for col in ['CPI', 'Unemployment']:
    combined[f'{col}_missing'] = combined[col].isna().astype('int8')
    combined[col] = combined.groupby('Store')[col].ffill()

train = combined[combined['source']=='train'].drop(columns='source').reset_index(drop=True)
test = combined[combined['source']=='test'].drop(columns='source').reset_index(drop=True)

# Cell 6 - Holiday type (not just True/False — each holiday behaves differently)


In [29]:
def holiday_type(date):
    d = date.strftime('%Y-%m-%d')
    superbowl = ['2010-02-12','2011-02-11','2012-02-10','2013-02-08']
    laborday  = ['2010-09-10','2011-09-09','2012-09-07','2013-09-06']
    thanksgiving = ['2010-11-26','2011-11-25','2012-11-23','2013-11-29']
    christmas = ['2010-12-31','2011-12-30','2012-12-28','2013-12-27']
    if d in superbowl: return 'SuperBowl'
    if d in laborday: return 'LaborDay'
    if d in thanksgiving: return 'Thanksgiving'
    if d in christmas: return 'Christmas'
    return 'Normal'

for df in [train, test]:
    df['holiday_type'] = df['Date'].apply(holiday_type)

# Cell 7 - Calendar + cyclical features


In [30]:
for df in [train, test]:
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['week_sin'] = np.sin(2*np.pi*df['week_of_year']/52)
    df['week_cos'] = np.cos(2*np.pi*df['week_of_year']/52)

# Cell 8 - Exact-date lag features (safe against gapped weeks, unlike .shift())


In [31]:
def add_lags(df, source, lag_weeks):
    for w in lag_weeks:
        shifted = source[['Store','Dept','Date','Weekly_Sales']].copy()
        shifted['Date'] = shifted['Date'] + pd.Timedelta(weeks=w)
        shifted = shifted.rename(columns={'Weekly_Sales': f'lag_{w}'})
        df = df.merge(shifted, on=['Store','Dept','Date'], how='left')
    return df

lag_weeks = [1, 2, 4, 13, 26, 51, 52, 53]
train = add_lags(train, train, lag_weeks)
test = add_lags(test, train, lag_weeks)  # test can only look back into train history

# Cell 9 - Rolling stats (computed on train history only, then joined to both)


In [32]:
# Cell 9 - Rolling stats (fixed for newer pandas)
roll_source = train[['Store','Dept','Date','Weekly_Sales']].sort_values(['Store','Dept','Date']).reset_index(drop=True).copy()
g = roll_source.groupby(['Store','Dept'])['Weekly_Sales']
roll_source['rolling_mean_4'] = g.shift(1).rolling(4).mean().reset_index(drop=True)
roll_source['rolling_median_8'] = g.shift(1).rolling(8).median().reset_index(drop=True)
roll_source['rolling_mean_13'] = g.shift(1).rolling(13).mean().reset_index(drop=True)
roll_source['rolling_mean_26'] = g.shift(1).rolling(26).mean().reset_index(drop=True)

roll_cols = ['Store','Dept','Date','rolling_mean_4','rolling_median_8','rolling_mean_13','rolling_mean_26']
train = train.merge(roll_source[roll_cols], on=['Store','Dept','Date'], how='left')

# for test, use the LAST available rolling value per Store-Dept (since test is future)
last_roll = roll_source.sort_values('Date').groupby(['Store','Dept'])[roll_cols[3:]].last().reset_index()
test = test.merge(last_roll, on=['Store','Dept'], how='left')

# Cell 10 - Trend + store-size normalized features


In [33]:
# Cell 10 - Trend + store-size normalized features
for df in [train, test]:
    df['sales_change_1'] = df['lag_1'] - df['lag_2']
    df['sales_per_store_size'] = df['lag_52'] / df['Size']
    df['markdown_per_store_size'] = df['markdown_total'] / df['Size']

# Cell 11 - Fallback aggregates for unseen/short Store-Dept combos (computed from TRAIN only)


In [34]:
# Cell 11 - Fallback aggregates for unseen/short Store-Dept combos (computed from TRAIN only)
dept_median = train.groupby('Dept')['Weekly_Sales'].median().rename('dept_median_sales')
store_median = train.groupby('Store')['Weekly_Sales'].median().rename('store_median_sales')
storedept_median = train.groupby(['Store','Dept'])['Weekly_Sales'].median().rename('storedept_median_sales')

for df_name, df in [('train', train), ('test', test)]:
    df = df.merge(dept_median, on='Dept', how='left')
    df = df.merge(store_median, on='Store', how='left')
    df = df.merge(storedept_median, on=['Store','Dept'], how='left')
    if df_name == 'train': train = df
    else: test = df

# Cell 12 - Save processed data (everyone loads from here, no need to redo this)


In [35]:
# Cell 12 - Save processed data (everyone loads from here, no need to redo this)
train.to_pickle('data/train_processed.pkl')
test.to_pickle('data/test_processed.pkl')
print(train.shape, test.shape)
print(train.isna().sum().sort_values(ascending=False).head(10))

(421570, 50) (115064, 50)
lag_53                  162969
lag_52                  160029
sales_per_store_size    160029
lag_51                  157173
lag_26                   84477
rolling_mean_26          81918
lag_13                   46533
rolling_mean_13          41750
rolling_median_8         25966
lag_4                    18792
dtype: int64


In [36]:
# ============================================================
# MLflow - log the preprocessing / feature-engineering pipeline
# ============================================================
import os
import tempfile

# The pipeline as it is actually executed in the cells above, in order.
PREPROCESSING_STEPS = [
    {'step': 'load_raw',
     'detail': 'read train/test/features/stores, parse Date, sort by Store-Dept-Date'},
    {'step': 'merge',
     'detail': "features on ['Store','Date','IsHoliday'] + stores on ['Store'], validate='many_to_one'"},
    {'step': 'markdown_missing_indicators',
     'detail': 'MarkDown*_missing flags, fillna(0), markdown_total, markdown_count, has_any_markdown'},
    {'step': 'macro_forward_fill',
     'detail': 'CPI / Unemployment missing flags, then per-Store ffill over train+test combined'},
    {'step': 'holiday_type',
     'detail': 'SuperBowl / LaborDay / Thanksgiving / Christmas / Normal'},
    {'step': 'calendar_features',
     'detail': 'year, month, week_of_year, week_sin, week_cos'},
    {'step': 'exact_date_lags',
     'detail': 'date-shifted merges (gap safe); test only looks back into train history'},
    {'step': 'rolling_statistics',
     'detail': 'rolling_mean_4 / rolling_median_8 / rolling_mean_13 / rolling_mean_26 from train history; '
               'test gets the last available value per Store-Dept'},
    {'step': 'trend_and_size_features',
     'detail': 'sales_change_1, sales_per_store_size, markdown_per_store_size'},
    {'step': 'fallback_aggregates',
     'detail': 'dept / store / store-dept median sales, computed on train only'},
    {'step': 'persist',
     'detail': 'data/train_processed.pkl, data/test_processed.pkl'},
]

with mlflow.start_run(run_name='feature_engineering') as run:
    mlflow.set_tags({
        'stage': 'preprocessing',
        'notebook': 'feature_engineering.ipynb',
        'dataset': 'walmart-recruiting-store-sales-forecasting',
    })

    mlflow.log_params({
        'lag_weeks': lag_weeks,
        'markdown_cols': markdown_cols,
        'rolling_windows': [4, 8, 13, 26],
        'forward_filled_cols': ['CPI', 'Unemployment'],
        'merge_validate': 'many_to_one',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })

    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'test_rows': test.shape[0],
        'test_cols': test.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'test_missing_cells': int(test.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
        'weekly_sales_mean': float(train['Weekly_Sales'].mean()),
        'weekly_sales_median': float(train['Weekly_Sales'].median()),
    })

    # The pipeline definition + the resulting schema.
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')

    # Dataset lineage: schema and profile only. The .pkl files are ~150 MB, too
    # large to push to the artifact store on every run.
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )
    mlflow.log_input(
        mlflow.data.from_pandas(test, name='test_processed'),
        context='testing',
    )

    with tempfile.TemporaryDirectory() as tmp:
        missing = train.isna().sum().sort_values(ascending=False).to_frame('missing')
        missing['missing_pct'] = missing['missing'] / len(train) * 100
        missing.to_csv(os.path.join(tmp, 'train_missing_report.csv'))

        train.dtypes.astype(str).to_frame('dtype').to_csv(os.path.join(tmp, 'train_dtypes.csv'))
        train.head(1000).to_csv(os.path.join(tmp, 'train_processed_sample.csv'), index=False)
        test.head(1000).to_csv(os.path.join(tmp, 'test_processed_sample.csv'), index=False)

        for fname in os.listdir(tmp):
            mlflow.log_artifact(os.path.join(tmp, fname), artifact_path='preprocessing')

    print('Logged preprocessing run:', run.info.run_id)

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference tim

Logged preprocessing run: 9bf639c8038041079bca764c27e33e26
🏃 View run feature_engineering at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/0/runs/9bf639c8038041079bca764c27e33e26
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/0
